<a href="https://colab.research.google.com/github/iadicarlo/seavigil/blob/main/notebooks/sentinel1_vessel_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SeaVigil: Sentinel-1 vessel detection (Allen Institute model)

Runs the open, pre-trained **Allen Institute** Sentinel-1 vessel detector (the model behind Skylight, Apache-2.0) on a Copernicus scene, on a free Colab GPU. Output is a `detections.csv` (lat/lon, length, heading, speed, fishing-vessel class) that feeds SeaVigil's explanation + authorization + evidence layer.

**Before you run:**
1. `Runtime` menu -> `Change runtime type` -> **GPU**.
2. Click the **key icon** (left sidebar) -> add two secrets with notebook access ON:
   - `CDSE_S3_KEY`  = your Copernicus S3 **access key**
   - `CDSE_S3_SECRET` = your Copernicus S3 **secret key**
3. `Runtime` -> `Run all`. First run takes a few minutes (weights + a ~2 GB scene).


## 1. Clone the model + pull the weights (Git LFS)


In [1]:
!sudo apt-get -qq update >/dev/null && sudo apt-get -qq install -y git-lfs >/dev/null
!git lfs install >/dev/null
import os
if not os.path.isdir('/content/vessel-detection-sentinels'):
    !git clone --depth 1 https://github.com/allenai/vessel-detection-sentinels.git /content/vessel-detection-sentinels
%cd /content/vessel-detection-sentinels
# pull only the Sentinel-1 detector + attribute weights + backbones
!git lfs pull --include='data/model_artifacts/sentinel-1/**' --include='torch_weights/**'
!ls -la data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445/best.pth data/model_artifacts/sentinel-1/attr/c34aa37/best.pth


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Cloning into '/content/vessel-detection-sentinels'...
remote: Enumerating objects: 281, done.
remote: Counting objects: 100% (281/281), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 281 (delta 50), reused 249 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (281/281), 10.36 MiB | 18.32 MiB/s, done.
Resolving deltas: 100% (50/

## 2. Install dependencies
Uses Colab's GPU PyTorch; installs GDAL matched to the system lib, plus the geo + model deps.


In [4]:
!sudo apt-get -qq install -y gdal-bin libgdal-dev >/dev/null
import subprocess
gdal_ver = subprocess.check_output(['gdal-config','--version']).decode().strip()
!pip -q install GDAL==$gdal_ver rasterio pyproj scikit-image lightgbm s2cloudless mapcalc python-dateutil boto3
import torch, torchvision
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available(), '| torchvision', torchvision.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.4/240.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
torch 2.11.0+cu128 | GPU available: True | torchvision 0.26.0+cu128


## 3. Download a Sentinel-1 scene from Copernicus (S3)
Defaults to the Galapagos scene from 2026-06-27. To target another reserve/date, find a scene at https://browser.dataspace.copernicus.eu (Sentinel-1, GRD, IW) and paste its `.SAFE` name + S3 date path below.


In [5]:
from google.colab import userdata
import boto3, pathlib
AK = userdata.get('CDSE_S3_KEY'); SK = userdata.get('CDSE_S3_SECRET')
s3 = boto3.client('s3', endpoint_url='https://eodata.dataspace.copernicus.eu',
                  aws_access_key_id=AK, aws_secret_access_key=SK)

SCENE = 'S1D_IW_GRDH_1SDV_20260627T114116_20260627T114145_003421_006075_6C48.SAFE'
S3_PREFIX = 'Sentinel-1/SAR/IW_GRDH_1S/2026/06/27/' + SCENE

dest = pathlib.Path('data') / SCENE
n = 0
for page in s3.get_paginator('list_objects_v2').paginate(Bucket='eodata', Prefix=S3_PREFIX):
    for obj in page.get('Contents', []):
        rel = obj['Key'][len(S3_PREFIX)+1:]
        out = dest / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file('eodata', obj['Key'], str(out))
        n += 1
print('downloaded', n, 'files ->', dest)
!du -sh {dest}

downloaded 26 files -> data/S1D_IW_GRDH_1SDV_20260627T114116_20260627T114145_003421_006075_6C48.SAFE
1.9G	data/S1D_IW_GRDH_1SDV_20260627T114116_20260627T114145_003421_006075_6C48.SAFE


## 4. Run inference (GPU)


In [17]:
import os
import sys
import pathlib

# 1. Setup the directory structure for the hardcoded config path
root_path = '/content/vessel-detection-sentinels'
!mkdir -p {root_path}/src/src/config
!ln -sf {root_path}/src/config/config.yml {root_path}/src/src/config/config.yml

# 2. Add src to path for imports to work
if f"{root_path}/src" not in sys.path:
    sys.path.append(f"{root_path}/src")
if root_path not in sys.path:
    sys.path.append(root_path)

# 3. Import and run the inference directly in the notebook process
from src.main import run_inference

# Execute
run_inference(
    raw_path='data/',
    scratch_path='data/scratch/',
    output_path='data/output/',
    detector_model_dir='data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445',
    postprocess_model_dir='data/model_artifacts/sentinel-1/attr/c34aa37',
    scene_id=SCENE,
    conf=0.85,
    nms_thresh=10.0,
    save_crops=False,
    catalog='sentinel1'
)

/content/vessel-detection-sentinels/src/main.py:70: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
INFO:     Started server process [5255]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:5557 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [5255]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/runners.py", line 195, in run
    return runner.run(main)
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "uvloop/loop.pyx", line 1512, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 15

## 5. Review + download the detections
Download `seavigil_detections.csv`, then hand it to SeaVigil's converter (`scripts/sar_detections_to_incidents.py`) to fold into the map with jurisdiction + authorization + evidence.


In [ ]:
import pandas as pd, glob
found = glob.glob('data/output/**/predictions.csv', recursive=True)
if not found:
    print('No predictions.csv found - check the inference cell output for errors.')
else:
    df = pd.read_csv(found[0])
    df['scene_id'] = SCENE
    print(len(df), 'vessel detections in', SCENE)
    cols = [c for c in ['lon','lat','score','vessel_length_m','vessel_speed_k','is_fishing_vessel'] if c in df.columns]
    display(df[cols].head(25))
    df.to_csv('seavigil_detections.csv', index=False)
    from google.colab import files; files.download('seavigil_detections.csv')
